In [0]:
from Model_call_chat import agent_call
from Model_call_embeddings import embedding_call
import json, mlflow, math
from mlflow.deployments import get_deploy_client

In [0]:
user = spark.sql("select current_user()").first()[0]
mlflow.set_experiment("/Workspace/Users/johnnybrave1996@gmail.com/Agentic AI Bootcamp 1.0/07 Portway Logistics/")

In [0]:
spark.sql("""
create or replace function agentic_ai.portway_logistics.parcel_status(parcel_id string)
returns string
comment "returns the current delivery status for a parcel, given its parcel id"
return coalesce((select max(status) from agentic_ai.portway_logistics.parcels p 
where p.parcel_id = parcel_status.parcel_id), "No parcel found with that id")
""")


spark.sql("""
create or replace function agentic_ai.portway_logistics.depot_contact(depot string)
returns string
comment "returns the contact phone number for a depot, given the depot name"
return coalesce((select max(phone) from agentic_ai.portway_logistics.depots d 
where d.depot = depot_contact.depot), "No depot found with that name")
""")

In [0]:
@mlflow.trace
def parcel_status(parcel_id):
    return spark.sql(f"Select agentic_ai.portway_logistics.parcel_status(:id) as r", args={"id":parcel_id}).first()["r"]

@mlflow.trace
def depot_contact(depot):
    return spark.sql(f"Select agentic_ai.portway_logistics.depot_contact(:d) as r", args={"d":depot}).first()["r"]

In [0]:
def score(a,b):
    dot = 0
    size_a = 0
    size_b = 0
    for i in range(len(a)):
        dot = dot + a[i] * b[i]
        size_a = size_a + a[i] * a[i]
        size_b = size_b + b[i] * b[i]
    return dot / (math.sqrt(size_a) * math.sqrt(size_b))


articles = []
for row in spark.table("agentic_ai.portway_logistics.help_articles").collect():
    articles.append({"title": row["title"], "text": row["text"], "embedding": embedding_call(row["text"])})

@mlflow.trace
def search_help(question):
    q = embedding_call(question)
    best = articles[0]
    for article in articles:
        if score(q, article["embedding"]) > score(q, best["embedding"]):
            best = article
    return best["text"]

print(search_help("What does in transit mean?"))

In [0]:
def docs_only(question):
    article = search_help(question)
    return agent_call("You are Portway's customer assistant. Answer from the help article.",f"article: {article} question: {question}" )

print(docs_only("Where is parcel PX-44717"))

In [0]:
tools = [
    {"type": "function", "function":{
        "name": "search_help",
        "description" : "Search Portway's help document. Use for questions about policies, delivery days or what a status means.",
        "parameters" : {"type": "object", "properties": {"question": {"type": "string"}}, "required": ["question"]
                       }}},
    
    {"type": "function", "function":{
        "name": "parcel_status",
        "description" : "Get the live delivery status of one parcel, given its parcel id, such as PX-4471. The status names the depot that has it.",
        "parameters" : {"type": "object", "properties": {"parcel_id": {"type": "string"}}, "required": ["parcel_id"]
                       }}},
    
    {"type": "function", "function":{
        "name": "depot_contact",
        "description" : "Get the phone number of a depot, given the depot name exactly as it appears in parcel status, such as Reading depot.",
        "parameters" : {"type": "object", "properties": {"depot": {"type": "string"}}, "required": ["depot"]
                       }}}
]

system_message = "You are Portway's customer assistant. Answer only from what the tools return. If tool says no parcel was found, say the parcel was not found."

In [0]:
def run_tool(name, arguments):
    if name == "search_help":
        return search_help(arguments["question"])
    if name == "parcel_status":
        return parcel_status(arguments["parcel_id"])
    if name == "depot_contact":
        return depot_contact(arguments["depot"])
    return "Unknown tool"

In [0]:
def agent(question):
    client = get_deploy_client("databricks")

    messages = [   
                {"role": "system", "content": system_message},
                {"role": "user", "content": question}
                ]
    used = []
    reply = client.predict(endpoint="gemini-free-tier",
                        inputs = {
                            "messages": messages,
                            "tools": tools
                        })["choices"][0]["message"]

    while reply.get("tool_calls"):
        messages.append(reply) 

        for call in reply["tool_calls"]:
            name = call["function"]["name"]
            used.append(name)
            result = run_tool(name, json.loads(call["function"]["arguments"]))
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})

        reply = client.predict(endpoint="gemini-free-tier",
                            inputs = {
                                "messages": messages,
                                "tools": tools
                            })["choices"][0]["message"]
    
    return reply["content"], ",".join(used)

In [0]:
questions = spark.table("agentic_ai.portway_logistics.questions").orderBy("question_id").collect()

results = []

for question in questions:
    before = docs_only(question["question"])
    after, used = agent(question["question"])
    results.append({"question_id": question["question_id"],"question": question["question"],"before_answer" : before,"after_answer":after,"tools_used": used,"expected_tools": question["expected_tools"], "expected_text": question["expected_text"]})
    
print(len(results))

In [0]:
spark.createDataFrame(results).select("question_id","question", "tools_used", "expected_tools", "expected_text", "before_answer", "after_answer",)\
    .write.format("delta").mode("overwrite").saveAsTable("agentic_ai.portway_logistics.answers")
display(spark.table("agentic_ai.portway_logistics.answers"))